In [1]:
!pip install --upgrade protobuf==3.20.3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 4.0 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.0
    Uninstalling protobuf-6.33.0:
      Successfully uninstalled protobuf-6.33.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
onnx 1.18.0 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
a2a-sdk 0.3.10 requires protobuf>=5.29.5, but you have protobuf 3.20.3 which is incompatible.
pydrive2 1.21.3 requires cryptography<44, but you have cryptography 46.0.3 which is incompatible.
pydrive

In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.sequence import pad_sequences
import pickle
import warnings
warnings.filterwarnings('ignore')

2026-04-28 02:51:18.001622: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-28 02:51:18.012937: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777344678.026566      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777344678.031966      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-28 02:51:18.049605: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [3]:
print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

# Variable 'test' renamed to 'df_test_data'
df_test_data = pd.read_csv('/kaggle/input/lmsys-chatbot-arena/test.csv')

# Variable 'sample_submission' renamed to 'df_sample_submission'
df_sample_submission = pd.read_csv('/kaggle/input/lmsys-chatbot-arena/sample_submission.csv')

print("Test Shape:", df_test_data.shape)
print("Sample Submission Shape:", df_sample_submission.shape)

TensorFlow Version: 2.18.0
GPU Available: []
Test Shape: (5748, 4)
Sample Submission Shape: (5748, 4)


2026-04-28 02:51:24.741488: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [4]:
df_test_data['text_concatenated'] = df_test_data['prompt'] + ' [SEP] ' + df_test_data['response_a'] + ' [SEP] ' + df_test_data['response_b']

In [5]:
SEQUENCE_LENGTH = 512

print("\n" + "="*80)
print("LOADING GRU MODELS AND TOKENIZER")
print("="*80)

with open('/kaggle/input/lmsys-hemkesh-10040-gru/tokenizer_gru.pkl', 'rb') as file_handler:
    text_tokenizer = pickle.load(file_handler)

test_sequences = text_tokenizer.texts_to_sequences(df_test_data['text_concatenated'])
test_padded_inputs = pad_sequences(test_sequences, maxlen=SEQUENCE_LENGTH, padding='post', truncating='post')

print("X_test shape:", test_padded_inputs.shape)


LOADING GRU MODELS AND TOKENIZER


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/lmsys-hemkesh-10040-gru/tokenizer_gru.pkl'

In [6]:
import tensorflow as tf
from tensorflow import keras
import json
import h5py
import tempfile
import shutil
import os
import traceback

class NonZeroMask(keras.layers.Layer):
    def __init__(self, convert_to_float=True, **kwargs):
        super().__init__(**kwargs)
        self.convert_to_float = convert_to_float
    
    def call(self, input_tensor):
        if isinstance(input_tensor, (list, tuple)):
            input_tensor = input_tensor[0]
        
        output_tensor = tf.math.not_equal(input_tensor, 0)
        if self.convert_to_float:
            return tf.cast(output_tensor, tf.float32)
        return output_tensor
    
    def get_config(self):
        config = super().get_config()
        config.update({"cast_to_float": self.convert_to_float})
        return config
    
    @classmethod
    def from_config(cls, config):
        return cls(**config)

def load_model_with_custom_config(model_file_path):
    
    with h5py.File(model_file_path, 'r') as file_handler:
        architecture_config = json.loads(file_handler.attrs['model_config'])
        
        for network_layer in architecture_config['config']['layers']:
            if network_layer['class_name'] == 'NotEqual': # Checking against original class name in saved file
                for graph_node in network_layer.get('inbound_nodes', []):
                    if len(graph_node['args']) > 1 and isinstance(graph_node['args'][1], int):
                        graph_node['args'] = [graph_node['args'][0]]
    
    with tempfile.NamedTemporaryFile(suffix='.h5', delete=False) as temp_file_obj:
        temp_file_path = temp_file_obj.name
    
    shutil.copy2(model_file_path, temp_file_path)
    
    with h5py.File(temp_file_path, 'r+') as file_handler:
        del file_handler.attrs['model_config']
        file_handler.attrs['model_config'] = json.dumps(architecture_config)
    
    # Note: 'NotEqual' key remains because saved model expects that specific string
    loaded_model = keras.models.load_model(
        temp_file_path,
        custom_objects={'NotEqual': NonZeroMask}, 
        compile=False
    )
    
    os.remove(temp_file_path)
    
    return loaded_model

k_fold_count = 5
ensemble_predictions_list = []

for current_fold_index in range(k_fold_count):
    input_model_path = f'/kaggle/input/lmsys-hemkesh-10040-gru/gru_model_fold_{current_fold_index}.h5'
    print(f"Loading fold {current_fold_index} from {input_model_path}")
    try:
        current_gru_model = load_model_with_custom_config(input_model_path)
        
        # Using previously renamed 'test_padded_inputs'
        fold_output_probs = current_gru_model.predict(test_padded_inputs, batch_size=32, verbose=0)
        ensemble_predictions_list.append(fold_output_probs)
        print(f"Fold {current_fold_index} predictions shape:", fold_output_probs.shape)
        
    except Exception as error_msg:
        print(f"Failed to load fold {current_fold_index}: {repr(error_msg)}")
        traceback.print_exc()

if ensemble_predictions_list:
    final_ensemble_probabilities = tf.reduce_mean(ensemble_predictions_list, axis=0).numpy()
    print(f"\nFinal averaged predictions shape: {final_ensemble_probabilities.shape}")
else:
    print("\nNo models loaded successfully!")

Loading fold 0 from /kaggle/input/lmsys-hemkesh-10040-gru/gru_model_fold_0.h5
Failed to load fold 0: FileNotFoundError(2, "Unable to synchronously open file (unable to open file: name = '/kaggle/input/lmsys-hemkesh-10040-gru/gru_model_fold_0.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)")
Loading fold 1 from /kaggle/input/lmsys-hemkesh-10040-gru/gru_model_fold_1.h5
Failed to load fold 1: FileNotFoundError(2, "Unable to synchronously open file (unable to open file: name = '/kaggle/input/lmsys-hemkesh-10040-gru/gru_model_fold_1.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)")
Loading fold 2 from /kaggle/input/lmsys-hemkesh-10040-gru/gru_model_fold_2.h5
Failed to load fold 2: FileNotFoundError(2, "Unable to synchronously open file (unable to open file: name = '/kaggle/input/lmsys-hemkesh-10040-gru/gru_model_fold_2.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)")
Loading fold 

Traceback (most recent call last):
  File "/tmp/ipykernel_11/1637546368.py", line 71, in <cell line: 0>
    current_gru_model = load_model_with_custom_config(input_model_path)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_11/1637546368.py", line 35, in load_model_with_custom_config
    with h5py.File(model_file_path, 'r') as file_handler:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/h5py/_hl/files.py", line 564, in __init__
    fid = make_fid(name, mode, userblock_size, fapl, fcpl, swmr=swmr)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/h5py/_hl/files.py", line 238, in make_fid
    fid = h5f.open(name, flags, fapl=fapl)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "h5py/_objects.pyx", line 56, in h5py._objects.with_phil.wrapper
  File "h5py/_objects.pyx", line 57, in h5py._objects.with_phil.wrapper
  File "h5py/h5f.py

In [7]:
print("\n" + "="*80)
print("AVERAGING PREDICTIONS")
print("="*80)

final_ensemble_probabilities = np.mean(ensemble_predictions_list, axis=0)

print("Average predictions shape:", final_ensemble_probabilities.shape)


AVERAGING PREDICTIONS
Average predictions shape: ()


In [8]:
print("\n" + "="*80)
print("CREATING SUBMISSION")
print("="*80)

df_final_submission = df_sample_submission.copy()

df_final_submission['winner_model_a'] = final_ensemble_probabilities[:, 0]
df_final_submission['winner_model_b'] = final_ensemble_probabilities[:, 1]
df_final_submission['winner_tie'] = final_ensemble_probabilities[:, 2]

submission_file_path = '/kaggle/working/submission.csv'
df_final_submission.to_csv(submission_file_path, index=False)

print("Saved:", submission_file_path)
print("Shape:", df_final_submission.shape)


CREATING SUBMISSION


IndexError: invalid index to scalar variable.

In [9]:
print("\n" + "="*80)
print("SUBMISSION PREVIEW")
print("="*80)
print(df_final_submission.head(20))


SUBMISSION PREVIEW
            id  winner_model_a  winner_model_b  winner_tie
0   3297560222        0.333333        0.333333    0.333333
1   2556155375        0.333333        0.333333    0.333333
2   1793939629        0.333333        0.333333    0.333333
3   2562993561        0.333333        0.333333    0.333333
4   2111345399        0.333333        0.333333    0.333333
5   2118339565        0.333333        0.333333    0.333333
6   1247276303        0.333333        0.333333    0.333333
7   1825483321        0.333333        0.333333    0.333333
8   1108527003        0.333333        0.333333    0.333333
9   1314503909        0.333333        0.333333    0.333333
10  2154277953        0.333333        0.333333    0.333333
11  3762682310        0.333333        0.333333    0.333333
12  2409640027        0.333333        0.333333    0.333333
13  1050325306        0.333333        0.333333    0.333333
14  1564267187        0.333333        0.333333    0.333333
15  1415251431        0.333333      

In [10]:
print("\n" + "="*80)
print("PREDICTION STATISTICS")
print("="*80)
print(f"Model A - Mean: {final_ensemble_probabilities[:, 0].mean():.4f}, Std: {final_ensemble_probabilities[:, 0].std():.4f}, Min: {final_ensemble_probabilities[:, 0].min():.4f}, Max: {final_ensemble_probabilities[:, 0].max():.4f}")
print(f"Model B - Mean: {final_ensemble_probabilities[:, 1].mean():.4f}, Std: {final_ensemble_probabilities[:, 1].std():.4f}, Min: {final_ensemble_probabilities[:, 1].min():.4f}, Max: {final_ensemble_probabilities[:, 1].max():.4f}")
print(f"Tie     - Mean: {final_ensemble_probabilities[:, 2].mean():.4f}, Std: {final_ensemble_probabilities[:, 2].std():.4f}, Min: {final_ensemble_probabilities[:, 2].min():.4f}, Max: {final_ensemble_probabilities[:, 2].max():.4f}")


PREDICTION STATISTICS


IndexError: invalid index to scalar variable.

In [11]:
print("\n" + "="*80)
print("VERIFYING PROBABILITIES SUM TO 1")
print("="*80)

probability_row_sums = final_ensemble_probabilities.sum(axis=1)

print("Min sum:", float(probability_row_sums.min()))
print("Max sum:", float(probability_row_sums.max()))
print("Mean sum:", float(probability_row_sums.mean()))


VERIFYING PROBABILITIES SUM TO 1


AxisError: axis 1 is out of bounds for array of dimension 0

In [12]:
print("\n" + "="*80)
print("INFERENCE COMPLETED")
print("="*80)
print("Submission file saved:", submission_file_path)


INFERENCE COMPLETED


NameError: name 'submission_file_path' is not defined